# Gray–Scott SLD

Task 1 — unseen initial conditions.

This is the supplied single-fold experiment: trajectory 6 is held out for testing, seven remaining trajectories are sampled for training, and two are used for validation. All sets use LR starts 100, 150, 200, and 250. The source leaves Python’s random split unseeded; PyTorch and NumPy retain seed 1.

Generate the paired `uv` MAT files with `data_generation/Gray-Scott/Gray-Scott_data.ipynb` before running this notebook. The default data folder is `<repository>/data/gray_scott`; results go to `<repository>/results/task1_unseen_ic/gray_scott`. `SLD_DATA_ROOT` and `SLD_RESULTS_ROOT` override the two root folders.

The model keeps the supplied equations, architecture, hyperparameters, and trajectory slicing (`LR[:, 50::2]`, `HR[:, 200::2]`). The segment starts above apply to these **post-slice arrays**, which have 351 LR frames and 1,401 HR frames. The effective stored HR spacing in this pipeline is `dt = 10`, because the raw five-step spacing is decimated again. The manuscript describes its starts in the raw stored trajectories; the supplied code therefore does not exactly reproduce that time-window description. These slices are preserved in this release. It maps 25 two-channel 32×32 frames to 100 two-channel 256×256 frames. The training configuration uses 1,000 epochs and can require substantial GPU memory. CPU execution is supported by device selection but full training has not been validated here.

This is one experiment configuration, not the complete repeated-fold paper aggregate. Cell outputs are cleared. The optional evaluation cell requires the best checkpoint produced by training.


In [ ]:
from pathlib import Path
import os

# Locate the repository from the notebook folder or repository root.
_start = Path.cwd().resolve()
ROOT = next((p for p in (_start, *_start.parents)
             if (p / "README.md").is_file() and (p / "data_generation").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the SLD repository or one of its subfolders.")
DATA_ROOT = Path(os.environ.get("SLD_DATA_ROOT", str(ROOT / "data"))).expanduser().resolve()
RESULTS_ROOT = Path(os.environ.get("SLD_RESULTS_ROOT", str(ROOT / "results"))).expanduser().resolve()


## Model, losses, and dataset split

Run after dataset generation. The data loading and normalization retain the original pipeline.


In [ ]:
"""SLD for the two-dimensional Gray–Scott system."""
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import Variable
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import lr_scheduler
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as scio
import time
import os
import random
import math
from torch.nn.utils import weight_norm, spectral_norm


torch.manual_seed(1)
np.random.seed(1)
torch.set_default_dtype(torch.float32)


# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create directory structure
data_dir = DATA_ROOT / 'gray_scott'
model_dir = RESULTS_ROOT / 'task1_unseen_ic' / 'gray_scott' / 'checkpoints'
fig_dir = RESULTS_ROOT / 'task1_unseen_ic' / 'gray_scott' / 'figures'

# Create directories if they don't exist
model_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {data_dir.absolute()}")
print(f"Model directory: {model_dir.absolute()}")
print(f"Figure directory: {fig_dir.absolute()}")

# Verify data exists
if not data_dir.exists():
    raise FileNotFoundError(f"Data directory not found: {data_dir.absolute()}")

# Check for data files
data_files = list(data_dir.glob('2DGS_IC*.mat'))
print(f"Found {len(data_files)} data files")

lapl_op = [[[[    0,   0, -1/12,   0,     0],
             [    0,   0,   4/3,   0,     0],
             [-1/12, 4/3,    -5, 4/3, -1/12],
             [    0,   0,   4/3,   0,     0],
             [    0,   0, -1/12,   0,     0]]]]


def initialize_weights(module):

    c = 1
    if isinstance(module, nn.Conv2d):
        module.weight.data.uniform_(-c*np.sqrt(1 / (3 * 3 * 320)), 
            c*np.sqrt(1 / (3 * 3 * 320)))

    if isinstance(module, nn.Conv1d):
        module.weight.data.uniform_(-c*np.sqrt(1 / (3 * 3 * 320)), 
            c*np.sqrt(1 / (3 * 3 * 320)))
     
    elif isinstance(module, nn.Linear):
        module.bias.data.zero_()

class ShiftMean(nn.Module):
    """
    支持 x 形状:
      - [T,B,C,H,W]
      - [B,C,H,W]
    """
    def __init__(self, mean, std):
        super().__init__()
        mean = torch.as_tensor(mean).float().view(1, 1, -1, 1, 1)  # [1,1,C,1,1]
        std  = torch.as_tensor(std ).float().view(1, 1, -1, 1, 1)
        self.register_buffer("mean", mean)
        self.register_buffer("std", std)

    def forward(self, x, mode: str):
        if x.dim() == 4:
            # [B,C,H,W] -> [1,B,C,H,W] 复用同一套广播
            x_in = x.unsqueeze(0)
            squeeze_back = True
        else:
            x_in = x
            squeeze_back = False

        if mode == "sub":
            y = (x_in - self.mean) / (self.std + 1e-12)
        elif mode == "add":
            y = x_in * (self.std + 1e-12) + self.mean
        else:
            raise NotImplementedError

        return y.squeeze(0) if squeeze_back else y
        
def haar_iwt2x(ll, lh, hl, hh):
    """
    Inverse Haar wavelet synthesis (2x upsampling).
    ll/lh/hl/hh: [B, C, H, W] -> out: [B, C, 2H, 2W]
    """
    x00 = (ll + lh + hl + hh) * 0.5
    x01 = (ll - lh + hl - hh) * 0.5
    x10 = (ll + lh - hl - hh) * 0.5
    x11 = (ll - lh - hl + hh) * 0.5

    B, C, H, W = ll.shape
    out = torch.empty(B, C, 2 * H, 2 * W, device=ll.device, dtype=ll.dtype)
    out[:, :, 0::2, 0::2] = x00
    out[:, :, 0::2, 1::2] = x01
    out[:, :, 1::2, 0::2] = x10
    out[:, :, 1::2, 1::2] = x11
    return out

class PairwiseLearnedInterpolator(nn.Module):
    """
    对一对相邻 LR 帧 (x0, x1) 生成中间细分帧。
    输入:
        x0, x1: [B, C, H, W]
        alpha:  [B, 1, H, W] or [1,1,1,1]
    输出:
        y:      [B, C, H, W]
    """
    def __init__(self, channels, hidden=32):
        super().__init__()
        in_ch = channels * 2 + 1  # x0, x1, alpha map

        self.fuse = nn.Sequential(
            nn.Conv2d(in_ch, hidden, 3, padding=1, padding_mode='circular'),
            nn.GELU(),
            nn.Conv2d(hidden, hidden, 3, padding=1, padding_mode='circular'),
            nn.GELU()
        )

        self.w_head = nn.Conv2d(hidden, 2 * channels, 3, padding=1, padding_mode='circular')
        self.r_head = nn.Conv2d(hidden, channels, 3, padding=1, padding_mode='circular')

    def forward(self, x0, x1, alpha):
        B, C, H, W = x0.shape

        if alpha.dim() == 0:
            alpha = alpha.view(1, 1, 1, 1).to(x0)
        if alpha.shape[0] == 1 and B > 1:
            alpha = alpha.expand(B, 1, H, W)
        elif alpha.shape[-2:] == (1, 1):
            alpha = alpha.expand(B, 1, H, W)

        base = (1.0 - alpha) * x0 + alpha * x1

        feat = torch.cat([x0, x1, alpha], dim=1)
        feat = self.fuse(feat)

        w = self.w_head(feat)                      # [B, 2C, H, W]
        r = self.r_head(feat)                     # [B, C, H, W]

        w0, w1 = torch.chunk(w, 2, dim=1)
        w0 = torch.sigmoid(w0)
        w1 = torch.sigmoid(w1)

        out = w0 * x0 + w1 * x1 + r
        out = out + base
        return out
        
class HaarPyramidSynthesis(nn.Module):

    def __init__(self, out_channels: int, scale: int):
        super().__init__()
        assert scale >= 2 and (scale & (scale - 1) == 0), "scale must be power of 2"
        self.C = out_channels
        self.scale = scale
        self.L = int(math.log2(scale))

    def forward(self, packed: torch.Tensor) -> torch.Tensor:
        B, ch, H, W = packed.shape
        C = self.C
        assert ch == C * (self.scale ** 2), f"expected {C*(self.scale**2)} channels, got {ch}"

        idx = 0
        # coarsest (at base H,W): 4*C
        ll = packed[:, idx:idx + C]; idx += C
        lh = packed[:, idx:idx + C]; idx += C
        hl = packed[:, idx:idx + C]; idx += C
        hh = packed[:, idx:idx + C]; idx += C

        x = haar_iwt2x(ll, lh, hl, hh)  # [B,C,2H,2W]

        # finer levels
        for m in range(1, self.L):
            r = 2 ** m
            per = C * (r * r)

            lh_p = packed[:, idx:idx + per]; idx += per
            hl_p = packed[:, idx:idx + per]; idx += per
            hh_p = packed[:, idx:idx + per]; idx += per

            # unpack to current spatial size (H*r, W*r)
            lh = F.pixel_shuffle(lh_p, r)
            hl = F.pixel_shuffle(hl_p, r)
            hh = F.pixel_shuffle(hh_p, r)

            x = haar_iwt2x(x, lh, hl, hh)

        assert idx == ch
        return x


class WaveletSRHead(nn.Module):
    def __init__(self, in_ch, out_ch, scale, conv3x3_factory):
        super().__init__()
        self.proj = conv3x3_factory(in_ch, out_ch * (scale ** 2))
        self.synth = HaarPyramidSynthesis(out_channels=out_ch, scale=scale)

    def forward(self, feat):
        packed = self.proj(feat)
        return self.synth(packed)

class ConvNeXt_GRU_Cell(nn.Module):
    """
    ConvNeXt-style GRU cell used as a continuous-time GRU derivative:
        dh/dt = (1 - z) * (h_tilde - h) + alpha * Lap(h) + beta_x * Dx(h) + beta_y * Dy(h)

    Notes:
        - Dx, Dy, Lap are fixed depthwise conv operators
        - alpha, beta_x, beta_y are learnable per-channel parameters
    """
    def __init__(self, dim, custom_act=None):
        super().__init__()
        self.dim = dim

        # Gates: r and z together
        self.gate_conv = nn.Sequential(
            nn.Conv2d(dim * 2, dim * 2, kernel_size=7, padding=3, groups=dim * 2, padding_mode='circular'),
            nn.GroupNorm(1, dim * 2),
            nn.Conv2d(dim * 2, dim * 2, kernel_size=1),
            nn.Sigmoid()
        )

        # Candidate path (ConvNeXt-ish bottleneck)
        self.candidate_dw = nn.Conv2d(dim * 2, dim * 2, kernel_size=7, padding=3, groups=dim * 2, padding_mode='circular')
        self.candidate_norm = nn.GroupNorm(1, dim * 2)
        self.candidate_pw1 = nn.Conv2d(dim * 2, dim * 4, kernel_size=1)
        self.act = custom_act if custom_act is not None else nn.GELU()
        self.candidate_pw2 = nn.Conv2d(dim * 4, dim, kernel_size=1)

        # ------------------------------------------------------------------
        # Fixed differential operators: depthwise Dx, Dy, Lap
        # ------------------------------------------------------------------
        self.dx = nn.Conv2d(
            dim, dim, kernel_size=3, padding=1, groups=dim,
            bias=False, padding_mode='circular'
        )
        self.dy = nn.Conv2d(
            dim, dim, kernel_size=3, padding=1, groups=dim,
            bias=False, padding_mode='circular'
        )
        self.lap = nn.Conv2d(
            dim, dim, kernel_size=3, padding=1, groups=dim,
            bias=False, padding_mode='circular'
        )

        self._init_fixed_kernels()

        # freeze fixed operators
        for p in self.dx.parameters():
            p.requires_grad = False
        for p in self.dy.parameters():
            p.requires_grad = False
        for p in self.lap.parameters():
            p.requires_grad = False

        # ------------------------------------------------------------------
        # Learnable coefficients (per-channel)
        # shape: [1, C, 1, 1] for broadcasting
        # small init to keep this branch weak at start
        # ------------------------------------------------------------------
        self.alpha  = nn.Parameter(torch.zeros(1, dim, 1, 1))
        self.beta_x = nn.Parameter(torch.zeros(1, dim, 1, 1))
        self.beta_y = nn.Parameter(torch.zeros(1, dim, 1, 1))

    def _init_fixed_kernels(self):
        """
        Central-difference-like first derivatives and 5-point Laplacian.
        """
        device = self.dx.weight.device
        dtype = self.dx.weight.dtype

        dx_kernel = torch.tensor(
            [[0.0, 0.0, 0.0],
             [-0.5, 0.0, 0.5],
             [0.0, 0.0, 0.0]],
            dtype=dtype, device=device
        )

        dy_kernel = torch.tensor(
            [[0.0, -0.5, 0.0],
             [0.0,  0.0, 0.0],
             [0.0,  0.5, 0.0]],
            dtype=dtype, device=device
        )

        lap_kernel = torch.tensor(
            [[0.0,  1.0, 0.0],
             [1.0, -4.0, 1.0],
             [0.0,  1.0, 0.0]],
            dtype=dtype, device=device
        )

        dx_weight = torch.zeros_like(self.dx.weight.data)   # [C,1,3,3]
        dy_weight = torch.zeros_like(self.dy.weight.data)
        lap_weight = torch.zeros_like(self.lap.weight.data)

        dx_weight[:, 0, :, :] = dx_kernel
        dy_weight[:, 0, :, :] = dy_kernel
        lap_weight[:, 0, :, :] = lap_kernel

        with torch.no_grad():
            self.dx.weight.copy_(dx_weight)
            self.dy.weight.copy_(dy_weight)
            self.lap.weight.copy_(lap_weight)

    def forward(self, h, x_input):
        """
        h: [B, dim, H, W]
        x_input: [B, dim, H, W]
        """
        combined = torch.cat([h, x_input], dim=1)  # [B, 2*dim, H, W]

        gates = self.gate_conv(combined)
        r, z = torch.split(gates, self.dim, dim=1)

        h_reset = r * h
        combined_reset = torch.cat([h_reset, x_input], dim=1)

        out = self.candidate_dw(combined_reset)
        out = self.candidate_norm(out)
        out = self.candidate_pw1(out)
        out = self.act(out)
        out = self.candidate_pw2(out)  # h_tilde

        # Original ODE-GRU term
        dh_dt_main = (1 - z) * (out - h)

        # Fixed local differential basis branch
        dh_dt_basis = (
            self.alpha  * self.lap(h) +
            self.beta_x * self.dx(h) +
            self.beta_y * self.dy(h)
        )

        dh_dt = dh_dt_main + dh_dt_basis
        return dh_dt

class ODE_ConvGRU_Func(nn.Module):
    """
    ODE RHS wrapper for torchdiffeq.

    Supports two modes:
      - Controlled ODE (recommended for SR): uses a time-varying guidance U(t)
        built from the temporally-interpolated LR sequence.
            dh/dt = cell(h, U(t))
      - Autonomous ODE (your old behavior): uses constant x_input (e.g., zeros)
            dh/dt = cell(h, x_input)
    """
    def __init__(self, cell):
        super().__init__()
        self.cell = cell

        # Autonomous forcing (old)
        self.x_input = None          # [B,C,H,W]

        # Controlled forcing (new)
        self.U_seq = None            # [T_out,B,C,H,W]
        self.t0 = None
        self.t1 = None

    #@torch.no_grad()
    def set_control(self, U_seq: torch.Tensor, t_steps: torch.Tensor):
        """
        U_seq:   [T_out,B,C,H,W]  (C should match hidden dim)
        t_steps: [T_out] (monotonic)
        """
        self.U_seq = U_seq
        self.t0 = float(t_steps[0].item())
        self.t1 = float(t_steps[-1].item())

    def _U_at(self, t: torch.Tensor) -> torch.Tensor:
        U = self.U_seq
        if U is None:
            return None
    
        T = U.shape[0]
        tt = torch.clamp(t, min=self.t0, max=self.t1).to(device=U.device, dtype=U.dtype)
        s = (tt - self.t0) / (self.t1 - self.t0 + 1e-12) * (T - 1)
    
        i0 = torch.floor(s).clamp(0, T - 2).long()
        i1 = i0 + 1
        w = (s - i0.float()).view(1, 1, 1, 1)
    
        return (1 - w) * U[i0] + w * U[i1]

    def forward(self, t, h):
        # Controlled path if U_seq is set
        if self.U_seq is not None:
            u_t = self._U_at(t)
            return self.cell(h, u_t)

        # Fallback autonomous path (old)
        if self.x_input is None:
            self.x_input = torch.zeros_like(h)
        return self.cell(h, self.x_input)

class TemporalResidualRefiner(nn.Module):
    """
    对 [T,B,C,H,W] 在时间维做轻量残差修正
    """
    def __init__(self, channels, hidden=None, kernel_size=3):
        super().__init__()
        assert kernel_size % 2 == 1, "kernel_size should be odd"
        hidden = hidden or max(16, channels * 4)
        pad = kernel_size // 2

        self.in_proj = nn.Conv1d(channels, hidden, kernel_size=1)
        self.tconv1 = nn.Conv1d(hidden, hidden, kernel_size=kernel_size, padding=pad)
        self.act = nn.GELU()
        self.tconv2 = nn.Conv1d(hidden, hidden, kernel_size=kernel_size, padding=pad)
        self.out_proj = nn.Conv1d(hidden, channels, kernel_size=1)

    def forward(self, x):
        """
        x: [T,B,C,H,W]
        """
        T, B, C, H, W = x.shape

        z = x.permute(1, 3, 4, 2, 0).contiguous()   # [B,H,W,C,T]
        z = z.view(B * H * W, C, T)                 # [BHW,C,T]

        z0 = z
        z = self.in_proj(z)
        z = self.act(self.tconv1(z))
        z = self.act(self.tconv2(z))
        z = self.out_proj(z)

        z = z + z0

        z = z.view(B, H, W, C, T)
        z = z.permute(4, 0, 3, 1, 2).contiguous()   # [T,B,C,H,W]
        return z
        
class temporal_sr(nn.Module):
    """
    方案 B:
    先线性插值，再做 temporal conv residual refinement
    输入:  [T, B, C, H, W]
    输出: [T*factor, B, C, H, W]
    """
    def __init__(self, t_upscale_factor, channels=2, hidden=32, kernel_size=5):
        super().__init__()
        self.t_upscale_factor = t_upscale_factor
        self.refiner = TemporalResidualRefiner(
            channels=channels,
            hidden=hidden,
            kernel_size=kernel_size
        )

    def forward(self, x):
        t, b, c, h, w = x.shape

        z = x.permute(1, 3, 4, 2, 0)                 # [B,H,W,C,T]
        z = z.contiguous().view(b * h * w, c, t)
        z = F.interpolate(
            z,
            scale_factor=self.t_upscale_factor,
            mode='linear',
            align_corners=True
        )
        z = z.contiguous().view(b, h, w, c, t * self.t_upscale_factor)
        z = z.permute(4, 0, 3, 1, 2).contiguous()   # [T',B,C,H,W]

        z = self.refiner(z)
        return z



class PhySR_ODE_ConvGRU(nn.Module):
    def __init__(self, n_feats, n_layers, upscale_factor,
                 shift_mean_paras_lr, shift_mean_paras_hr, custom_act=None):
        super().__init__()
        self.n_next_blocks, self.n_resblock = n_layers
        self.t_up_factor, self.s_up_factor = upscale_factor

        mean_lres, std_lres = shift_mean_paras_lr
        mean_hres, std_hres = shift_mean_paras_hr

        self.tsr = temporal_sr(self.t_up_factor)

        # Encoder
        self.feat_extract = nn.Conv2d(2, n_feats, kernel_size=3, padding=1, padding_mode='circular')
        self.h0_proj = nn.Conv2d(n_feats, n_feats, kernel_size=1)

        # ODE-ConvGRU (你原封不动保留)
        self.gru_cell = ConvNeXt_GRU_Cell(n_feats, custom_act=custom_act)
        self.ode_wrapper = ODE_ConvGRU_Func(self.gru_cell)
        
        def wn_circ_conv3x3(in_ch, out_ch):
            return weight_norm(nn.Conv2d(in_ch, out_ch, 3, padding=1, padding_mode='circular'))

        self.tail = WaveletSRHead(n_feats, out_ch=2, scale=self.s_up_factor, conv3x3_factory=wn_circ_conv3x3)
        self.skip = WaveletSRHead(2,      out_ch=2, scale=self.s_up_factor, conv3x3_factory=wn_circ_conv3x3)


        self.shift_in = ShiftMean(mean_lres, std_lres)
        self.shift_out = ShiftMean(mean_hres, std_hres)

    def forward(self, x, t_steps=None, odeint=None, method='rk4', use_control=True):
        """
        x: [T_in, B, 2, H, W]
        Returns: [T_out, B, 2, H*s_up, W*s_up]
        """
        T_in, B, C, H, W = x.shape
        x = self.shift_in(x, mode='sub')

        # 1) Temporal interpolation in LR space
        x_lr_up = self.tsr(x)                 # [T_out,B,2,H,W]
        T_out = x_lr_up.shape[0]

        # 2) time grid
        if t_steps is None:
            t_steps = torch.linspace(0, float(T_out - 1), T_out, device=x.device, dtype=x.dtype)
        else:
            t_steps = t_steps.to(device=x.device, dtype=x.dtype)
            if t_steps.numel() != T_out:
                raise ValueError(f"t_steps length {t_steps.numel()} must equal T_out {T_out}")

        # 3) Build guidance U_seq
        x_flat = x_lr_up.reshape(T_out * B, C, H, W)
        U_flat = self.feat_extract(x_flat)                 # [T_out*B,n_feats,H,W]
        U_seq = U_flat.view(T_out, B, -1, H, W)            # [T_out,B,n_feats,H,W]

        h = self.h0_proj(U_seq[0])
        hs = []
        dt = t_steps[1] - t_steps[0]
        for k in range(T_out):
            # dh = self.gru_cell(h, U_seq[k])
            # h = h + dt * dh
            k1 = self.gru_cell(h, U_seq[k])
            h_mid = h + 0.5 * dt * k1
            k2 = self.gru_cell(h_mid, U_seq[k])
            h = h + dt * k2

            hs.append(h)

        h_stack = torch.stack(hs, dim=0)                   # [T_out,B,n_feats,H,W]

        # ✅ 4) Vectorized spatial SR (把 T 并到 batch，一次性做完)
        h_flat = h_stack.reshape(T_out * B, -1, H, W)       # [T_out*B,n_feats,H,W]
        xlr_flat = x_lr_up.reshape(T_out * B, 2, H, W)      # [T_out*B,2,H,W]

        y_flat = self.tail(h_flat) + self.skip(xlr_flat)    # [T_out*B,2,H*s,W*s]
        y = y_flat.view(T_out, B, 2, H*self.s_up_factor, W*self.s_up_factor)

        y = self.shift_out(y, mode='add')
        return y


class Conv2dDerivative(nn.Module):
    def __init__(self, DerFilter, resol, kernel_size=3, name=''):
        super(Conv2dDerivative, self).__init__()

        self.resol = resol  # constant in the finite difference
        self.name = name
        self.input_channels = 1
        self.output_channels = 1
        self.kernel_size = kernel_size

        self.padding = int((kernel_size - 1) / 2)
        self.filter = nn.Conv2d(self.input_channels, self.output_channels, self.kernel_size, 
            1, padding=0, bias=False)

        # Fixed gradient operator
        self.filter.weight = nn.Parameter(torch.FloatTensor(DerFilter), requires_grad=False)  

    def forward(self, input):
        derivative = self.filter(input)
        return derivative / self.resol


class Conv1dDerivative(nn.Module):
    def __init__(self, DerFilter, resol, kernel_size=3, name=''):
        super(Conv1dDerivative, self).__init__()

        self.resol = resol  # $\delta$*constant in the finite difference
        self.name = name
        self.input_channels = 1
        self.output_channels = 1
        self.kernel_size = kernel_size

        self.padding = int((kernel_size - 1) / 2)
        self.filter = nn.Conv1d(self.input_channels, self.output_channels, self.kernel_size, 
            1, padding=0, bias=False)
        # Fixed gradient operator
        self.filter.weight = nn.Parameter(torch.FloatTensor(DerFilter), requires_grad=False)  

    def forward(self, input):
        derivative = self.filter(input)
        return derivative / self.resol


class LossGenerator(nn.Module):
    '''Calculate the physics loss and the data loss'''

    def __init__(self, dt = (10.0/200), dx = (20.0/128)):
       
        super(LossGenerator, self).__init__()

        # spatial derivative operator
        self.laplace = Conv2dDerivative(
            DerFilter = lapl_op,
            resol = (dx**2),
            kernel_size = 5,
            name = 'laplace_operator').to(device)

        # forward/backward derivative operator 
        self.dt = Conv1dDerivative(
            DerFilter = [[[-1/2, 0, 1/2]]],
            resol = (dt),
            kernel_size = 3,
            name = 'partial_t').to(device) 

        self.fwd_dt = Conv1dDerivative(
            DerFilter = [[[-3/2, 2, -1/2]]],
            resol = (dt),
            kernel_size = 3,
            name = 'forward_partial_t').to(device) 
 
        self.bwd_dt = Conv1dDerivative(
            DerFilter = [[[1/2, -2, 3/2]]],
            resol = (dt),
            kernel_size = 3,
            name = 'backward_partial_t').to(device) 

    def GetPhyLoss(self, output):
        '''Calculate the physical loss'''
        # output: [t,b,c,h,w]

        ############### spatial derivatives #################
        # laplace u, [t-2,b,c,h,w]
        u = output[:, :, 0:1, :, :]
        len_t,len_b,len_c,len_h,len_w = u.shape 
        # [t,b,c,h,w] -> [t*b,c,h,w]
        u = u.reshape(len_t*len_b, len_c, len_h, len_w)
        laplace_u = self.laplace(u)  
        # change batch to [t,b,c,h,w]
        laplace_u = laplace_u.reshape(len_t,len_b,len_c,len_h-4,len_w-4)

        # laplace v, [t-2,b,c,h,w]
        v = output[:, :, 1:2, :, :]
        len_t,len_b,len_c,len_h,len_w = v.shape 
        v = v.reshape(len_t*len_b, len_c, len_h, len_w)
        laplace_v = self.laplace(v)  
        laplace_v = laplace_v.reshape(len_t,len_b,len_c,len_h-4,len_w-4)

        ############### temporal derivatives #################
        # u_t, [t,b,c,h-4,w-4]
        u = output[:, :, 0:1, 2:-2, 2:-2]
        len_t,len_b,len_c,len_h,len_w = u.shape 
        u = u.permute(3,4,1,2,0) # [h,w,b,c,t]
        u = u.reshape(len_h*len_w*len_b, len_c, len_t) # [h*w*b,c,t]
        u_t = self.dt(u) # [h*w*b,c,t-2]
        u_t0 = self.fwd_dt(u[:,:,0:3])
        u_tn = self.bwd_dt(u[:,:,-3:])
        u_t = torch.cat((u_t0,u_t,u_tn), dim=2) # [h*w*b,c,t]
        u_t = u_t.reshape(len_h, len_w, len_b, len_c, len_t)
        u_t = u_t.permute(4,2,3,0,1)

        # v_t, [t,b,c,h-4,w-4]
        v = output[:, :, 1:2, 2:-2, 2:-2]
        len_t,len_b,len_c,len_h,len_w = v.shape 
        v = v.permute(3,4,1,2,0) # [h,w,b,c,t]
        v = v.reshape(len_h*len_w*len_b, len_c, len_t) # [h*w*b,c,t]
        v_t = self.dt(v)
        v_t0 = self.fwd_dt(v[:,:,0:3])
        v_tn = self.bwd_dt(v[:,:,-3:])
        v_t = torch.cat((v_t0, v_t, v_tn), dim=2)
        v_t = v_t.reshape(len_h, len_w, len_b, len_c, len_t)
        v_t = v_t.permute(4,2,3,0,1)

        ############### corresponding u & v ###################
        u = output[:, :, 0:1, 2:-2, 2:-2]  # [step, b, c, height(Y), width(X)]
        v = output[:, :, 1:2, 2:-2, 2:-2]  # [step, b, c, height(Y), width(X)]

        # make sure the dimensions consistent
        assert laplace_u.shape == u_t.shape
        assert u_t.shape == v_t.shape
        assert laplace_u.shape == u.shape
        assert laplace_v.shape == v.shape

        # 2D GS
        Du, Dv = 0.16, 0.08
        f, k = 0.06, 0.062
        f_u = Du*laplace_u - u*v**2 + f*(1-u) - u_t
        f_v = Dv*laplace_v + u*v**2 - (f+k)*v - v_t

        return f_u, f_v


    def GetModelLoss(self, model):
        ''' Get the L2-norm of the model '''

        l2_reg = torch.tensor(0.).to(device)
        for param in model.parameters():
            l2_reg += torch.norm(param)

        return l2_reg


def LossGen(output, truth, beta, loss_func):

    L1_loss = nn.L1Loss()
    MSE_loss = nn.MSELoss()

    # data loss
    data_loss = L1_loss(output, truth)

    # phy loss, output shape: [t,b,c,h,w]
    output = torch.cat((output[:, :, :, :, -2:], output, output[:, :, :, :, 0:3]), dim=4)
    output = torch.cat((output[:, :, :, -2:, :], output, output[:, :, :, 0:3, :]), dim=3)
    
    f_u, f_v = loss_func.GetPhyLoss(output)
    phy_loss = MSE_loss(f_u, torch.zeros_like(f_u).to(device)) + MSE_loss(
                f_v, torch.zeros_like(f_v).to(device))

    loss = data_loss + beta * phy_loss 

    return loss, data_loss, phy_loss

def LossGenGS(output, truth, beta, loss_func=None, use_phy_loss=True, uv_beta=(1.0, 1.0)):
    """
    output, truth: [T,B,2,H,W]
    beta: global physics weight
    uv_beta: (w_u, w_v)
    """
    L1_loss = nn.L1Loss()
    MSE_loss = nn.MSELoss()

    data_loss = L1_loss(output, truth)

    if (not use_phy_loss) or (loss_func is None) or (beta <= 0.0):
        zero = torch.zeros(1, device=output.device, dtype=output.dtype)
        return data_loss, data_loss, zero, zero, zero

    # periodic padding
    output_pad = torch.cat((output[:, :, :, :, -2:], output, output[:, :, :, :, 0:3]), dim=4)
    output_pad = torch.cat((output_pad[:, :, :, -2:, :], output_pad, output_pad[:, :, :, 0:3, :]), dim=3)

    f_u, f_v = loss_func.GetPhyLoss(output_pad)

    phy_u = MSE_loss(f_u, torch.zeros_like(f_u))
    phy_v = MSE_loss(f_v, torch.zeros_like(f_v))
    phy_loss = uv_beta[0] * phy_u + uv_beta[1] * phy_v

    total_loss = data_loss + beta * phy_loss
    return total_loss, data_loss, phy_loss, phy_u, phy_v


def train_gs(model, train_loader, val_loader, n_iters, lr, print_every,
             dt, dx, beta, device,
             phy_start_epoch=300, beta_ramp_epochs=300,
             uv_beta=(1.0, 1.0),
             best_ckpt_path=None,
             hist_save_path=None,
             pretrain_flag=False, pretrain_ckpt_path=None):

    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_iters, eta_min=1e-7)

    # 如果你们已有 GS 的 LossGenerator，就继续用它
    loss_function = LossGenerator(dt, dx)

    if pretrain_flag and pretrain_ckpt_path is not None and os.path.exists(pretrain_ckpt_path):
        ckpt = torch.load(pretrain_ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        print(f"Loaded pretrain checkpoint from: {pretrain_ckpt_path}")

    best_error = 1e9
    mse = nn.MSELoss()

    hist = {
        "train_loss": [],
        "train_data_loss": [],
        "train_phy_loss": [],
        "train_phy_u": [],
        "train_phy_v": [],
        "val_loss": [],
        "val_data_loss": [],
        "val_phy_loss": [],
        "val_phy_u": [],
        "val_phy_v": [],
        "val_error": [],
        "lr": [],
        "beta_epoch": [],
        "mode": []
    }

    for epoch in range(n_iters):
        # warmup + ramp
        if epoch < phy_start_epoch:
            use_phy_loss = False
            beta_epoch = 0.0
            mode_now = "warmup_data_only"
        else:
            use_phy_loss = True
            ramp_k = min(1.0, (epoch - phy_start_epoch + 1) / float(beta_ramp_epochs))
            beta_epoch = beta * ramp_k
            mode_now = "data_plus_gs_phy"

        model.train()
        tr_tot = tr_data = tr_phy = tr_u = tr_v = 0.0
        nb = 0

        for lres, hres in train_loader:
            lres = lres.to(device).transpose(0, 1)
            hres = hres.to(device).transpose(0, 1)

            optimizer.zero_grad()
            out = model(lres)

            total_loss, data_loss, phy_loss, phy_u, phy_v = LossGenGS(
                out, hres, beta=beta_epoch, loss_func=loss_function,
                use_phy_loss=use_phy_loss, uv_beta=uv_beta
            )

            if torch.isnan(total_loss) or torch.isinf(total_loss):
                print(f"[Epoch {epoch+1}] NaN/Inf detected, skipping batch")
                optimizer.zero_grad()
                continue

            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            optimizer.step()

            tr_tot += total_loss.item()
            tr_data += data_loss.item()
            tr_phy += phy_loss.item()
            tr_u += phy_u.item()
            tr_v += phy_v.item()
            nb += 1

        scheduler.step()

        if nb == 0:
            print(f"[Epoch {epoch+1}] No valid training batches, skipped.")
            continue

        tr_tot_avg = tr_tot / nb
        tr_data_avg = tr_data / nb
        tr_phy_avg = tr_phy / nb
        tr_u_avg = tr_u / nb
        tr_v_avg = tr_v / nb

        # validation
        model.eval()
        va_tot = va_data = va_phy = va_u = va_v = va_rel = 0.0
        nv = 0

        with torch.no_grad():
            for lres, hres in val_loader:
                lres = lres.to(device).transpose(0, 1)
                hres = hres.to(device).transpose(0, 1)

                out = model(lres)

                total_loss, data_loss, phy_loss, phy_u, phy_v = LossGenGS(
                    out, hres, beta=beta_epoch, loss_func=loss_function,
                    use_phy_loss=use_phy_loss, uv_beta=uv_beta
                )

                rel_err = torch.sqrt(
                    mse(hres, out) / (mse(hres, torch.zeros_like(hres)) + 1e-12)
                )

                va_tot += total_loss.item()
                va_data += data_loss.item()
                va_phy += phy_loss.item()
                va_u += phy_u.item()
                va_v += phy_v.item()
                va_rel += rel_err.item()
                nv += 1

        if nv == 0:
            print(f"[Epoch {epoch+1}] No validation batches, skipped.")
            continue

        va_tot_avg = va_tot / nv
        va_data_avg = va_data / nv
        va_phy_avg = va_phy / nv
        va_u_avg = va_u / nv
        va_v_avg = va_v / nv
        va_rel_avg = va_rel / nv

        # record history
        hist["train_loss"].append(tr_tot_avg)
        hist["train_data_loss"].append(tr_data_avg)
        hist["train_phy_loss"].append(tr_phy_avg)
        hist["train_phy_u"].append(tr_u_avg)
        hist["train_phy_v"].append(tr_v_avg)

        hist["val_loss"].append(va_tot_avg)
        hist["val_data_loss"].append(va_data_avg)
        hist["val_phy_loss"].append(va_phy_avg)
        hist["val_phy_u"].append(va_u_avg)
        hist["val_phy_v"].append(va_v_avg)
        hist["val_error"].append(va_rel_avg)

        hist["lr"].append(optimizer.param_groups[0]['lr'])
        hist["beta_epoch"].append(beta_epoch)
        hist["mode"].append(mode_now)

        if (epoch + 1) % print_every == 0 or epoch == 0:
            print(f"[Epoch {epoch+1}/{n_iters}] mode={mode_now}, beta_epoch={beta_epoch:.3e}, lr={optimizer.param_groups[0]['lr']:.2e}")
            print(f"train: total={tr_tot_avg:.4e}, data={tr_data_avg:.4e}, phy={tr_phy_avg:.4e}, phy_u={tr_u_avg:.4e}, phy_v={tr_v_avg:.4e}")
            print(f"val  : total={va_tot_avg:.4e}, data={va_data_avg:.4e}, phy={va_phy_avg:.4e}, phy_u={va_u_avg:.4e}, phy_v={va_v_avg:.4e}, rel={va_rel_avg:.4e}")

        if va_rel_avg < best_error:
            best_error = va_rel_avg

            if best_ckpt_path is not None:
                torch.save({
                    "epoch": epoch + 1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "best_error": best_error,
                }, best_ckpt_path)

            print(f"New best checkpoint saved. best_rel_err={best_error:.6e}")

    hist["best_error"] = best_error
    hist["best_ckpt_path"] = best_ckpt_path

    if hist_save_path is not None:
        torch.save({
            "hist": hist
        }, hist_save_path)
        print(f"Full history saved to {hist_save_path}")

    return hist


def validate_gs(model, val_loader, loss_function, beta, device):
    model.eval()
    val_loss = 0.0
    val_error = 0.0
    mse_function = nn.MSELoss()

    with torch.no_grad():
        for lres, hres in val_loader:
            lres = lres.to(device).transpose(0, 1)
            hres = hres.to(device).transpose(0, 1)

            outputs = model(lres)
            loss, _, _, _, _ = LossGenGS(outputs, hres, beta, loss_function, use_phy_loss=(beta > 0))

            error = torch.sqrt(
                mse_function(hres, outputs.detach()) /
                (mse_function(hres, torch.zeros_like(hres)) + 1e-12)
            )

            val_loss += loss.item()
            val_error += error.item()

    val_error = val_error / len(val_loader)
    val_loss = val_loss / len(val_loader)

    return val_loss, val_error


def test_gs(model, test_loader, ckpt_path, save_dir, fig_save_dir, device):
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(fig_save_dir, exist_ok=True)

    mse_function = nn.MSELoss()
    pred_error = 0.0

    with torch.no_grad():
        for idx, (lres, hres) in enumerate(test_loader):
            lres = lres.to(device).transpose(0, 1)
            hres = hres.to(device).transpose(0, 1)

            outputs = model(lres)

            error = torch.sqrt(
                mse_function(hres, outputs.detach()) /
                (mse_function(hres, torch.zeros_like(hres)) + 1e-12)
            )
            pred_error += error.item()

            torch.save({
                "pred": outputs.detach().cpu(),
                "lres": lres.cpu(),
                "hres": hres.cpu()
            }, os.path.join(save_dir, f"output_{idx}.pt"))

            t = np.arange(hres.shape[0])
            for b in range(hres.shape[1]):
                u_pred = outputs[:, b, 0, :, :].detach().cpu().numpy()
                u_true = hres[:, b, 0, :, :].cpu().numpy()

                plt.figure()
                plt.plot(t, u_pred[:, 66, 66], label='u-pred')
                plt.plot(t, u_true[:, 66, 66], label='u-ref')
                plt.xlabel('t')
                plt.ylabel('u')
                plt.legend()
                plt.savefig(os.path.join(fig_save_dir, f'u_comp_[i={idx}][b={b}].png'))
                plt.close()

    pred_error = pred_error / len(test_loader)
    return pred_error


# ==================== DATASET CLASS ====================
class GSDataset(Dataset):
    def __init__(self, data_dir, data_fname, ICs, random_influence=0.1):
        self.samples = []
        self.data_dir = data_dir
        self.data_fname = data_fname

        # 和你原代码一致：0.1 -> ri10
        ri_tag = f"ri{int(random_influence * 100):02d}"

        for ic in ICs:
            lres_filename = f'{data_fname}{ic}_{ri_tag}_2x751x32x32.mat'
            hres_filename = f'{data_fname}{ic}_{ri_tag}_2x3001x256x256.mat'

            lres = scio.loadmat(os.path.join(data_dir, lres_filename))['uv']
            hres = scio.loadmat(os.path.join(data_dir, hres_filename))['uv']

            lres = lres[:, 50::2, ...]   # [2,351,32,32]
            hres = hres[:, 200::2, ...]  # [2,1401,256,256]

            lres = np.transpose(lres, (1, 0, 2, 3))  # [351,2,32,32]
            hres = np.transpose(hres, (1, 0, 2, 3))  # [1401,2,256,256]

            l_starts = [100, 150, 200, 250]
            for ls in l_starts:
                hs = ls * 4
                self.samples.append((
                    torch.tensor(lres[ls:ls+25], dtype=torch.float32),
                    torch.tensor(hres[hs:hs+100], dtype=torch.float32)
                ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


if __name__ == '__main__':
    data_fname = '2DGS_IC'
    RANDOM_INFLUENCE = 0.1

    ICs = np.arange(1, 11)

    # 固定 test IC，然后剩下随机划 train / val
    test_ic = 6
    remaining_ICs = [ic for ic in ICs if ic != test_ic]
    train_ICs = random.sample(remaining_ICs, 7)
    val_ICs = [ic for ic in remaining_ICs if ic not in train_ICs]
    test_ICs = [test_ic]

    print("train_ICs:", train_ICs)
    print("val_ICs:", val_ICs)
    print("test_ICs:", test_ICs)

    print("\nLoading datasets by IC split...")

    train_data = GSDataset(
        data_dir=data_dir,
        data_fname=data_fname,
        ICs=train_ICs,
        random_influence=RANDOM_INFLUENCE
    )

    val_data = GSDataset(
        data_dir=data_dir,
        data_fname=data_fname,
        ICs=val_ICs,
        random_influence=RANDOM_INFLUENCE
    )

    test_data = GSDataset(
        data_dir=data_dir,
        data_fname=data_fname,
        ICs=test_ICs,
        random_influence=RANDOM_INFLUENCE
    )

    if len(train_data) == 0 or len(val_data) == 0 or len(test_data) == 0:
        raise ValueError("One of train/val/test datasets is empty! Please check your data files.")

    print(f"Train samples: {len(train_data)}")
    print(f"Val samples  : {len(val_data)}")
    print(f"Test samples : {len(test_data)}")

    sample_lres, sample_hres = train_data[0]
    print(f"One sample LR shape: {sample_lres.shape}")
    print(f"One sample HR shape: {sample_hres.shape}")

    train_lres = torch.stack([train_data[i][0] for i in range(len(train_data))])
    train_hres = torch.stack([train_data[i][1] for i in range(len(train_data))])

    mean_lres = train_lres.mean(dim=(0, 1, 3, 4))
    std_lres  = train_lres.std(dim=(0, 1, 3, 4))
    mean_hres = train_hres.mean(dim=(0, 1, 3, 4))
    std_hres  = train_hres.std(dim=(0, 1, 3, 4))

    print(f"mean_lres: {mean_lres}")
    print(f"std_lres : {std_lres}")
    print(f"mean_hres: {mean_hres}")
    print(f"std_hres : {std_hres}\n")

    train_loader = DataLoader(train_data, batch_size=4, shuffle=True, num_workers=0)
    val_loader   = DataLoader(val_data, batch_size=4, shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_data, batch_size=4, shuffle=False, num_workers=0)

    print("\nBuilding model...")

    n_iters = 1000
    learning_rate = 5e-3
    print_every = 2
    dt = 1.0 * 5 * 2
    dx = 1.0
    steps = 100
    effective_step = list(range(0, steps))

    beta = 0.005
    phy_start_epoch = 300
    beta_ramp_epochs = 300

    print('Super-Resolution for 2D GS equation...')


## Train SLD

This cell creates the model, trains it, and saves the best validation checkpoint and full history.


In [ ]:
model = PhySR_ODE_ConvGRU(
    n_feats=64,
    n_layers=[6, 3],
    upscale_factor=[4, 8],
    shift_mean_paras_lr=[mean_lres, std_lres],
    shift_mean_paras_hr=[mean_hres, std_hres],
    custom_act=None
).to(device)

save_dir = model_dir
os.makedirs(save_dir, exist_ok=True)
best_ckpt_path = os.path.join(save_dir, "Gray-Scott_SLD_best_model.pth")
hist_save_path = os.path.join(save_dir, "Gray-Scott_SLD_full_history.pth")

def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

total, trainable = count_parameters(model)
print(f'{"-"*30}')
print(f'Total parameters: {total:,}')
print(f'Trainable parameters: {trainable:,}')
print(f'{"-"*30}')

print("\nStarting training...")
start = time.time()

hist = train_gs(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_iters=n_iters,
    lr=learning_rate,
    print_every=print_every,
    dt=dt,
    dx=dx,
    beta=beta,
    device=device,
    best_ckpt_path=best_ckpt_path,
    hist_save_path=hist_save_path
)

train_loss_list = hist["train_loss"]
val_loss_list = hist["val_loss"]
val_error_list = hist["val_error"]

end = time.time()
print(f'Training time: {(end - start):.2f} seconds')
print(f'Best val rel error: {hist["best_error"]:.6e}')


## Optional test evaluation

This optional evaluation has not been executed during repository preparation. Requires a trained, trusted checkpoint. Set `RUN_TEST_EVALUATION = True` to export predictions and the original `u` point time-series plots. The reported number is the supplied helper’s batch-average error; it is not the paper’s complete fold aggregation.


In [ ]:
# Optional: evaluate all 100 target frames using a trained checkpoint.
RUN_TEST_EVALUATION = False
if RUN_TEST_EVALUATION:
    if not Path(best_ckpt_path).is_file():
        raise FileNotFoundError(f"Train first or supply a trusted checkpoint: {best_ckpt_path}")
    test_error = test_gs(
        model, test_loader, best_ckpt_path,
        model_dir.parent / "predictions", fig_dir, device
    )
    print(f"Test relative L2 error (batch average): {test_error:.6e}")
